In [1]:
# ==========================================
# =                 SETUP                  = 
# ==========================================
import sys
import os
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np

if os.path.abspath('..') not in sys.path:
    sys.path.append(os.path.abspath('..'))

from src.dataset import get_data_splits, get_dataloaders
from src.augmentation import get_valid_transform_clahe
from src.models import get_resnet18_modified, get_efficientnet_b0_modified, get_cct_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("A preparar dados...")

# Carregar o mesmo Val_Loader que usaste na Análise
TRAIN_DIR = "../data/GTSRB/Training"
train_df, val_df = get_data_splits(TRAIN_DIR)

_, val_loader = get_dataloaders(
    train_df, val_df, 
    train_transform=get_valid_transform_clahe(), 
    val_transform=get_valid_transform_clahe(), 
    batch_size=64
)

A preparar dados...


In [2]:
# ==========================================
# =      CARREGAR O TOP 3 DE MODELOS       =
# ==========================================
print("A carregar os pesos dos 3 melhores modelos...")

# Modelo A: EfficientNet Oclusão
model_a = get_efficientnet_b0_modified(num_classes=43, pretrained=False).to(device)
model_a.load_state_dict(torch.load("../checkpoints/modelo4_oclusao_efficientnet.pth", map_location=device))
model_a.eval()

# Modelo B: EfficientNet MixUp
model_b = get_efficientnet_b0_modified(num_classes=43, pretrained=False).to(device)
model_b.load_state_dict(torch.load("../checkpoints/modelo5_mixup_efficientnet.pth", map_location=device))
model_b.eval()

# Modelo C: CCT Transformer
model_c = get_cct_model(num_classes=43, pretrained=False).to(device)
model_c.load_state_dict(torch.load("../checkpoints/modelo6_cct_transformer.pth", map_location=device))
model_c.eval()

A carregar os pesos dos 3 melhores modelos...


VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 192, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((192,), eps=1e-06, elementwise_affine=True, bias=True)
      (attn): Attention(
        (qkv): Linear(in_features=192, out_features=576, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): Linear(in_features=192, out_features=192, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): LayerNorm((192,), eps=1e-06, elementwise_affine=True, bias=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=192, out_features=768, bias=True)
        (act): GELU(approximate='none')
        (drop1): Dropout(

In [5]:
# ==========================================
# =    CALCULAR O SOFT VOTING (ENSEMBLE)   =
# ==========================================
print(f"A processar as {len(val_df)} imagens.")

correct_ensemble = 0
correct_a, correct_b, correct_c = 0, 0, 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        
        # 1. Obter as previsões cruas de cada modelo
        logits_a = model_a(images)
        logits_b = model_b(images)
        logits_c = model_c(images)
        
        # 2. Converter para Probabilidades (0 a 1) usando Softmax
        probs_a = F.softmax(logits_a, dim=1)
        probs_b = F.softmax(logits_b, dim=1)
        probs_c = F.softmax(logits_c, dim=1)
        
        # 3. O ENSEMBLE (Soft Voting): Fazemos a média das opiniões dos 3 modelos
        ensemble_probs = (probs_a + probs_b + probs_c) / 3.0
        
        # 4. Ver qual é a classe com maior probabilidade
        _, pred_a = torch.max(probs_a, 1)
        _, pred_b = torch.max(probs_b, 1)
        _, pred_c = torch.max(probs_c, 1)
        _, pred_ensemble = torch.max(ensemble_probs, 1)
        
        # 5. Contar os acertos
        total += labels.size(0)
        correct_a += pred_a.eq(labels).sum().item()
        correct_b += pred_b.eq(labels).sum().item()
        correct_c += pred_c.eq(labels).sum().item()
        correct_ensemble += pred_ensemble.eq(labels).sum().item()

print("\n" + "="*45)
print("= RESULTADOS FINAIS (CONJUNTO DE VALIDAÇÃO) =")
print("="*45)
print(f"Modelo 4 Individual (Oclusão):   {100. * correct_a / total:.2f}%")
print(f"Modelo 5 Individual (MixUp):     {100. * correct_b / total:.2f}%")
print(f"Modelo 6 Individual (ViT):       {100. * correct_c / total:.2f}%")
print("-" * 45)
print(f" ENSEMBLE (O Poder dos 3):     {100. * correct_ensemble / total:.2f}%")
print("="*45)

A processar as 5303 imagens.

= RESULTADOS FINAIS (CONJUNTO DE VALIDAÇÃO) =
Modelo 4 Individual (Oclusão):   99.98%
Modelo 5 Individual (MixUp):     99.94%
Modelo 6 Individual (ViT):       99.87%
---------------------------------------------
 ENSEMBLE (O Poder dos 3):     99.98%
